In [1]:
print("hello")

hello


In [21]:
import os
import re
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

load_dotenv()

DOC_PATH = "2511.18538v5.pdf"


loader = PyPDFLoader(DOC_PATH)
pages = loader.load()    

print(f"Loaded {len(pages)} pages")

full_text = "\n".join(p.page_content for p in pages)

doc = Document(
    page_content=full_text,
    metadata={"source": DOC_PATH}
)

# --------------------------------------------------
# 2. Chunking (paper-safe)
# --------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n\n", "\n\n", "\n", ". ", " "]
)

chunks = text_splitter.split_documents([doc])
print(f"Raw chunks: {len(chunks)}")

# --------------------------------------------------
# 3. Chunk filters
# --------------------------------------------------
def is_to_chunk(text: str) -> bool:
    lines = text.splitlines()
    dot_lines = sum(bool(re.search(r"\.{2,}\s*\d+$", l)) for l in lines)
    numbered_lines = sum(bool(re.match(r"\d+(\.\d+)*\s+", l)) for l in lines)
    return dot_lines >= 3 or numbered_lines >= 5

def is_figure_chunk(text: str) -> bool:
    t = text.strip().lower()
    return t.startswith("figure") and len(t) < 800

def is_too_short(text: str) -> bool:
    return len(text) < 400

clean_chunks = []
for c in chunks:
    t = c.page_content
    if is_to_chunk(t):
        continue
    if is_figure_chunk(t):
        continue
    if is_too_short(t):
        continue
    clean_chunks.append(c)

chunks = clean_chunks
print(f"Clean chunks: {len(chunks)}")

# --------------------------------------------------
# 4. Vector store
# --------------------------------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

vectorstore.save_local("faiss_index")

# --------------------------------------------------
# 5. Retrieval
# --------------------------------------------------
QUESTION = "What is the main focus and contribution of this survey paper?"

RETRIEVAL_SCORE_THRESHOLD = 0.85

results = vectorstore.similarity_search_with_score(
    QUESTION,
    k=8
)

# Print distance range
if results:
    distances = [score for _, score in results]
    print(f"\nDistance range: {min(distances):.3f} to {max(distances):.3f}")

def is_junk_chunk(text: str) -> bool:
    if len(text) < 300:
        return True
    if "@" in text:
        return True
    return False

filtered = []
for doc, score in results:
    if score < RETRIEVAL_SCORE_THRESHOLD:
        continue
    if is_junk_chunk(doc.page_content):
        continue
    filtered.append((doc, score))

filtered.sort(key=lambda x: x[1])
selected = filtered[:4]

if not selected:
    raise RuntimeError("No reliable context retrieved")

# --------------------------------------------------
# 6. Build context with explicit IDs
# --------------------------------------------------
context_blocks = []
for i, (doc, _) in enumerate(selected):
    context_blocks.append(
        f"[DOC {i+1}]\n{doc.page_content.strip()}"
    )

context = "\n\n".join(context_blocks)

# --------------------------------------------------
# 7. LLM call (hallucination-ALLOWED prompt)
# --------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

messages = [
    (
        "system",
        "You answer questions using the provided documents. "
        "If a statement is supported by a document, cite it using [DOC X]. "
        "If the documents do not clearly support a statement, "
        "still answer but mark it as [UNSUPPORTED]."
    ),
    (
        "user",
        f"""
Documents:
{context}

Question:
{QUESTION}

Rules:
- Answer in plain language
- Every factual statement must end with either:
  • a citation like [DOC 1]
  • or the tag [UNSUPPORTED]
- Do not refuse to answer
- Do not explain your reasoning
- Do not summarize documents

Answer:
"""
    )
]

response = llm.invoke(messages)
print(response.content)


def extract_claims(text: str):
    prompt = f"""
Extract ONLY factual claims about the world or the document content from the text below.

Definition:
A valid claim is a declarative statement that asserts a fact about:
- entities, events, dates, locations, or properties described in the document

Do NOT extract:
- statements about missing information
- statements about lack of evidence
- statements explaining why the answer is unsupported
- meta statements like "the answer is unsupported" or "the documents do not provide information"

Instructions:
- Return each valid claim on a separate line
- Keep claims atomic
- If there are NO valid factual claims, return ONLY the word: NONE

Text:
\"\"\"{text}\"\"\"
"""

    response = llm.invoke(
        [{"role": "user", "content": prompt}]
    )

    output = response.content.strip()

    if output == "NONE":
        return []

    return [c.strip() for c in output.split("\n") if c.strip()]

claims = extract_claims(response.content)

print("\nEXTRACTED CLAIMS:\n")


verifier_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def fact_check_claim(claim: str, context: str) -> float:
    prompt = f"""
You are a strict fact-checker.

Claim:
{claim}

Context:
{context}

Task:
Determine whether the claim is supported by the context.

Rules:
- Use ONLY the information explicitly stated or clearly implied in the context
- Do NOT use outside knowledge
- Do NOT assume missing facts
- If the context is related but does not directly support the claim, return 0.5
- If the context clearly supports the claim, return 1.0
- If the context does not support the claim, return 0.0

Return ONLY a single number: 1.0, 0.5, or 0.0
"""
    response = verifier_llm.invoke(
        [{"role": "user", "content": prompt}]
    )
    return float(response.content.strip())


all_max_scores = []  # Create this BEFORE the loop

for i, claim in enumerate(claims, 1):
    print(f"{i}. {claim}")
    search = vectorstore.similarity_search_with_score(claim, k=3)  
    
    scores = []
    for doc, dist in search:
        checking = fact_check_claim(claim, doc.page_content)
        scores.append(checking)
    
    max_score = max(scores)
    all_max_scores.append(max_score)  # Save it!
    
    if max_score >= 1.0:
        print(f"  ✓ SUPPORTED - {claim} (score: {max_score})")
    elif max_score >= 0.5:
        print(f"  ⚠ PARTIAL - {claim} (score: {max_score})")
    else:
        print(f"  ✗ HALLUCINATED - {claim} (score: {max_score})")
    print()

# NOW the summary will work
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total claims: {len(claims)}")
print(f"✓ Supported: {sum(1 for s in all_max_scores if s >= 1.0)}")
print(f"⚠ Partial: {sum(1 for s in all_max_scores if 0.5 <= s < 1.0)}")
print(f"✗ Hallucinated: {sum(1 for s in all_max_scores if s < 0.5)}")

Loaded 303 pages
Raw chunks: 1488
Clean chunks: 1432

Distance range: 1.090 to 1.254
The main focus of this survey paper is to provide a comprehensive synthesis of research literature on large language models (LLMs) for code intelligence. It examines various aspects of LLMs, including their applications in code generation, software engineering, and code summarization, while also addressing technical elements like pretraining and supervised fine-tuning. The survey aims to serve as a reference for researchers and a roadmap for practitioners in the field [DOC 1]. Additionally, it evaluates benchmarks and methodologies for assessing code quality and identifies emerging trends and challenges in code generation systems [DOC 2].

EXTRACTED CLAIMS:

1. The main focus of this survey paper is to provide a comprehensive synthesis of research literature on large language models (LLMs) for code intelligence.
  ✓ SUPPORTED - The main focus of this survey paper is to provide a comprehensive synthesis

In [24]:
import os
import re
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langsmith import expect


# --------------------------------------------------
# 1. Load PDF (KEEP STRUCTURE)
# --------------------------------------------------

def load_document(doc_path):
    try:
        load_dotenv()
    except Exception as e:
        print(f"Error loading .env file: {e}")

    try:
        DOC_PATH = doc_path
        loader = PyPDFLoader(DOC_PATH)
        pages = loader.load()
        return pages 

    except Exception as e:
        print(f"Error loading document: {e}")
        return None
    
# --------------------------------------------------
# 2. Light page cleanup (safe)
# --------------------------------------------------

def clean_pages(pages: list[Document]) -> list[Document]:
    for p in pages:
        text = p.page_content
        text = re.sub(r"\n{3,}", "\n\n", text)   # collapse excessive newlines
        text = re.sub(r"\s+", " ", text)         # normalize whitespace
        p.page_content = text.strip()
    return pages

# --------------------------------------------------
# 3. Chunk WITHIN pages (token-aware)
# --------------------------------------------------

def chunk_text(pages):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50,
        separators=["\n\n", "\n"]
    )

    chunks = text_splitter.split_documents(pages)
    
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
        c.metadata["char_len"] = len(c.page_content)

    return chunks

PDF_PATH = "2511.18538v5.pdf"   # ← CHANGE PDF HERE ONLY

pages = load_document(PDF_PATH)
pages = clean_pages(pages)
chunks = chunk_text(pages)

# --------------------------------------------------
# 5. INSPECT CHUNKS (MANDATORY)
# --------------------------------------------------

print(f"\nTotal pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}\n")

for i, c in enumerate(chunks[:5]):  # preview first 5 chunks
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print(f"Metadata: {c.metadata}")
    print("-" * 80)
    print(c.page_content)
    print()


Total pages: 303
Total chunks: 303

CHUNK 1
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:4177c2c)', 'creationdate': '', 'author': 'Jian Yang; Xianglong Liu; Weifeng Lv; Ken Deng; Shawn Guo; Lin Jing; Yizhi Li; Shark Liu; Xianzhen Luo; Yuyu Luo; Changzai Pan; Ensheng Shi; Yingshui Tan; Renshuai Tao; Jiajun Wu; Xianjie Wu; Zhenhe Wu; Daoguang Zan; Chenchen Zhang; Wei Zhang; He Zhu; Terry Yue Zhuo; Kerui Cao; Xianfu Cheng; Jun Dong; Shengjie Fang; Zhiwei Fei; Xiangyuan Guan; Qipeng Guo; Zhiguang Han; Joseph James; Tianqi Luo; Renyuan Li; Yuhang Li; Yiming Liang; Congnan Liu; Jiaheng Liu; Qian Liu; Ruitong Liu; Tyler Loakman; Xiangxin Meng; Chuang Peng; Tianhao Peng; Jiajun Shi; Mingjie Tang; Boyang Wang; Haowen Wang; Yunli Wang; Fanglin Xu; Zihan Xu; Fei Yuan; Ge Zhang; Jiayi Zhang; Xinhao Zhang; Wangchunshu Zhou; Hualei Zhu; King Zhu; Bryan Dai; Aishan Liu; Zhoujun Li; Chenghua Lin; Tianyu Liu; Chao Peng; Kai Shen; Libo Qin; Shuangyong Song; Zizheng Zhan; J

In [ ]:
PDF_PATH = "hallu_sample.pdf"   # ← CHANGE PDF HERE ONLY

pages = load_document(PDF_PATH)
pages = clean_pages(pages)
chunks = chunk_text(pages)

PDF_PATH = "hallu_sample.pdf"   # ← CHANGE PDF HERE ONLY

pages = load_document(PDF_PATH)
pages = clean_pages(pages)
chunks = chunk_text(pages)


# --------------------------------------------------
# 5. INSPECT CHUNKS (MANDATORY)
# --------------------------------------------------

print(f"\nTotal pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}\n")

for i, c in enumerate(chunks[:5]):  # preview first 5 chunks
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print(f"Metadata: {c.metadata}")
    print("-" * 80)
    print(c.page_content)
    print()

In [ ]:
"""
PDF to Clean Chunks - Minimal Version
Only: Load → Clean → Chunk → Return cleaned chunks ready for embeddings
"""

import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document



# --------------------------------------------------
# 1. Load PDF
# --------------------------------------------------
def load_document(doc_path):
    try:
        load_dotenv()
    except Exception as e:
        print(f"Error loading .env file: {e}")

    try:
        DOC_PATH = doc_path
        loader = PyPDFLoader(DOC_PATH)
        pages = loader.load()
        return pages 

    except Exception as e:
        print(f"Error loading document: {e}")
        return None

# --------------------------------------------------
# 2. Clean Pages
# --------------------------------------------------
def clean_pages(pages):
    for p in pages:
        text = p.page_content
        
        # Remove PDF metadata
        text = re.sub(r'producer:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creator:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creationdate:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'ptex\.fullbanner:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'trapped:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'arxivid:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'doi:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'license:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'title:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        
        # Remove author lists
        text = re.sub(r'author:\s*.*?(?=\n\n|\Z)', '', text, flags=re.IGNORECASE | re.DOTALL)
        
        # Remove arXiv identifiers
        text = re.sub(r'arXiv:\d+\.\d+v\d+\s+\[.*?\]\s+\d+\s+\w+\s+\d{4}', '', text)
        
        # Remove page numbers
        text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
        
        # Remove metadata blocks and separators
        text = re.sub(r"Metadata:\s*\{[^}]*\}", '', text)
        text = re.sub(r'-{40,}', '', text)
        text = re.sub(r'={40,}', '', text)
        
        # Normalize whitespace
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r' \n', '\n', text)
        
        p.page_content = text.strip()
    
    # Filter short pages
    pages = [p for p in pages if len(p.page_content) > 100]
    return pages

# --------------------------------------------------
# 3. Clean Individual Chunks
# --------------------------------------------------
def clean_chunk_content(text):
    # Remove chunk artifacts
    text = re.sub(r'CHUNK\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'chunk_id[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'char_len[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(page|page_label|source|total_pages)[:\s]+[^\s,}]+', '', text)
    text = re.sub(r'[-=]{4,}', '', text)
    text = re.sub(r'\[\d+\]\s*$', '', text)
    
    # Final cleanup
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# --------------------------------------------------
# 4. Chunk Text
# --------------------------------------------------
def chunk_text(pages, chunk_size=800, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = text_splitter.split_documents(pages)
    
    # Clean and add metadata
    for i, c in enumerate(chunks):
        c.page_content = clean_chunk_content(c.page_content)
        c.metadata["chunk_id"] = i
        c.metadata["char_len"] = len(c.page_content)
        c.metadata["word_count"] = len(c.page_content.split())
    
    # Filter low-quality chunks
    chunks = [c for c in chunks if 
              len(c.page_content) > 50 and 
              not c.page_content.lower().startswith('metadata') and
              len(c.page_content.split()) > 5]
    
    return chunks

# --------------------------------------------------
# 5. Vector store
# --------------------------------------------------

def chunk_embeddings(chunks):
    embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")

    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    vectorstore.save_local("faiss_index")
    return vectorstore


# --------------------------------------------------
# 6. Retrieval
# --------------------------------------------------
QUESTION = "where is Bath city located?"

RETRIEVAL_SCORE_THRESHOLD = 0.85

results = vectorstore.similarity_search_with_score(
    QUESTION,
    k=8
)

def is_junk_chunk(text: str) -> bool:
    if len(text) < 300:
        return True
    if "@" in text:
        return True
    return False

filtered = []
for doc, score in results:
    if score < RETRIEVAL_SCORE_THRESHOLD:
        continue
    if is_junk_chunk(doc.page_content):
        continue
    filtered.append((doc, score))

filtered.sort(key=lambda x: x[1])
selected = filtered[:4]

if not selected:
    raise RuntimeError("No reliable context retrieved")


# --------------------------------------------------
# 6. Build context with explicit IDs
# --------------------------------------------------
context_blocks = []
for i, (doc, _) in enumerate(selected):
    context_blocks.append(
        f"[DOC {i+1}]\n{doc.page_content.strip()}"
    )

context = "\n\n".join(context_blocks)

# --------------------------------------------------
# 7. LLM call (hallucination-ALLOWED prompt)
# --------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

messages = [
    (
        "system",
        "You answer questions using the provided documents. "
        "If a statement is supported by a document, cite it using [DOC X]. "
        "If the documents do not clearly support a statement, "
        "still answer but mark it as [UNSUPPORTED]."
    ),
    (
        "user",
        f"""
Documents:
{context}

Question:
{QUESTION}

Rules:
- Answer in plain language
- Every factual statement must end with either:
  • a citation like [DOC 1]
  • or the tag [UNSUPPORTED]
- Do not refuse to answer
- Do not explain your reasoning
- Do not summarize documents

Answer:
"""
    )
]

response = llm.invoke(messages)
print(response.content)


def extract_claims(text: str):
    prompt = f"""
Extract ONLY factual claims about the world or the document content from the text below.

Definition:
A valid claim is a declarative statement that asserts a fact about:
- entities, events, dates, locations, or properties described in the document

Do NOT extract:
- statements about missing information
- statements about lack of evidence
- statements explaining why the answer is unsupported
- meta statements like "the answer is unsupported" or "the documents do not provide information"

Instructions:
- Return each valid claim on a separate line
- Keep claims atomic
- If there are NO valid factual claims, return ONLY the word: NONE

Text:
\"\"\"{text}\"\"\"
"""

    response = llm.invoke(
        [{"role": "user", "content": prompt}]
    )

    output = response.content.strip()

    if output == "NONE":
        return []

    return [c.strip() for c in output.split("\n") if c.strip()]

claims = extract_claims(response.content)

print("\nEXTRACTED CLAIMS:\n")


verifier_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def fact_check_claim(claim: str, context: str) -> float:
    prompt = f"""
You are a strict fact-checker.

Claim:
{claim}

Context:
{context}

Task:
Determine whether the claim is supported by the context.

Rules:
- Use ONLY the information explicitly stated or clearly implied in the context
- Do NOT use outside knowledge
- Do NOT assume missing facts
- If the context is related but does not directly support the claim, return 0.5
- If the context clearly supports the claim, return 1.0
- If the context does not support the claim, return 0.0

Return ONLY a single number: 1.0, 0.5, or 0.0
"""
    response = verifier_llm.invoke(
        [{"role": "user", "content": prompt}]
    )
    return float(response.content.strip())


all_max_scores = []  # Create this BEFORE the loop

for i, claim in enumerate(claims, 1):
    print(f"{i}. {claim}")
    search = vectorstore.similarity_search_with_score(claim, k=3)  
    
    scores = []
    for doc, dist in search:
        checking = fact_check_claim(claim, doc.page_content)
        scores.append(checking)
    
    max_score = max(scores)
    all_max_scores.append(max_score)  # Save it!
    
    if max_score >= 1.0:
        print(f"  ✓ SUPPORTED - {claim} (score: {max_score})")
    elif max_score >= 0.5:
        print(f"  ⚠ PARTIAL - {claim} (score: {max_score})")
    else:
        print(f"  ✗ HALLUCINATED - {claim} (score: {max_score})")
    print()



In [ ]:
# --------------------------------------------------
# MAIN - Process PDF to Clean Chunks
# --------------------------------------------------
PDF_PATH = "2511.18538v5.pdf"  # ← Change your PDF path here

# Load
pages = load_document(PDF_PATH)
print(f"Loaded {len(pages)} pages")

# Clean
pages = clean_pages(pages)
print(f"After cleaning: {len(pages)} pages")

# Chunk
chunks = chunk_text(pages, chunk_size=800, chunk_overlap=100)
print(f"Created {len(chunks)} chunks")

# Statistics
print(f"\nAvg chars per chunk: {sum(c.metadata['char_len'] for c in chunks) / len(chunks):.0f}")
print(f"Avg words per chunk: {sum(c.metadata['word_count'] for c in chunks) / len(chunks):.0f}")

# Preview first 3 chunks
print(f"\n{'='*80}")
print("FIRST 3 CHUNKS:")
print(f"{'='*80}\n")

for i, c in enumerate(chunks[:3]):
    print(f"CHUNK {i+1} | Page: {c.metadata.get('page', 'N/A')} | Words: {c.metadata['word_count']}")
    print("-" * 80)
    print(c.page_content[:400] + "..." if len(c.page_content) > 400 else c.page_content)
    print("\n")

# Now 'chunks' is ready for embeddings
# Example: 
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings()
# vectorstore = FAISS.from_documents(chunks, embeddings)

In [7]:
%pip install faiss-cpu
%pip install langchain-openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
"""
Complete RAG Pipeline with Hallucination Detection
Load → Clean → Chunk → Embed → Retrieve → Answer → Fact-Check
"""


import re
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

# Load environment variables
load_dotenv()

# --------------------------------------------------
# 1. Load PDF
# --------------------------------------------------
def load_document(doc_path):
    try:
        loader = PyPDFLoader(doc_path)
        pages = loader.load()
        print(f"✓ Loaded {len(pages)} pages from {doc_path}")
        return pages 
    except Exception as e:
        print(f"✗ Error loading document: {e}")
        return None

# --------------------------------------------------
# 2. Clean Pages
# --------------------------------------------------
def clean_pages(pages):
    print("Cleaning pages...")
    for p in pages:
        text = p.page_content
        
        # Remove PDF metadata
        text = re.sub(r'producer:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creator:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creationdate:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'ptex\.fullbanner:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'trapped:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'arxivid:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'doi:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'license:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'title:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        
        # Remove author lists
        text = re.sub(r'author:\s*.*?(?=\n\n|\Z)', '', text, flags=re.IGNORECASE | re.DOTALL)
        
        # Remove arXiv identifiers
        text = re.sub(r'arXiv:\d+\.\d+v\d+\s+\[.*?\]\s+\d+\s+\w+\s+\d{4}', '', text)
        
        # Remove page numbers
        text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
        
        # Remove metadata blocks and separators
        text = re.sub(r"Metadata:\s*\{[^}]*\}", '', text)
        text = re.sub(r'-{40,}', '', text)
        text = re.sub(r'={40,}', '', text)
        
        # Normalize whitespace
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r' \n', '\n', text)
        
        p.page_content = text.strip()
    
    # Filter short pages
    pages = [p for p in pages if len(p.page_content) > 100]
    print(f"✓ Cleaned {len(pages)} pages")
    return pages

# --------------------------------------------------
# 3. Clean Individual Chunks
# --------------------------------------------------
def clean_chunk_content(text):
    # Remove chunk artifacts
    text = re.sub(r'CHUNK\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'chunk_id[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'char_len[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(page|page_label|source|total_pages)[:\s]+[^\s,}]+', '', text)
    text = re.sub(r'[-=]{4,}', '', text)
    text = re.sub(r'\[\d+\]\s*$', '', text)
    
    # Final cleanup
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# --------------------------------------------------
# 4. Chunk Text
# --------------------------------------------------
def chunk_text(pages, chunk_size=800, chunk_overlap=100):
    print(f"Chunking text (size={chunk_size}, overlap={chunk_overlap})...")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = text_splitter.split_documents(pages)
    
    # Clean and add metadata
    for i, c in enumerate(chunks):
        c.page_content = clean_chunk_content(c.page_content)
        c.metadata["chunk_id"] = i
        c.metadata["char_len"] = len(c.page_content)
        c.metadata["word_count"] = len(c.page_content.split())
    
    # Filter low-quality chunks
    chunks = [c for c in chunks if 
              len(c.page_content) > 50 and 
              not c.page_content.lower().startswith('metadata') and
              len(c.page_content.split()) > 5]
    
    print(f"✓ Created {len(chunks)} chunks")
    return chunks

# --------------------------------------------------
# 5. Create or Load Vector Store
# --------------------------------------------------
def create_vectorstore(chunks, index_path="faiss_index"):
    """Create new vector store from chunks"""
    print("Creating embeddings and vector store...")
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    
    vectorstore.save_local(index_path)
    print(f"✓ Vector store saved to {index_path}")
    return vectorstore

def load_vectorstore(index_path="faiss_index"):
    """Load existing vector store"""
    print(f"Loading vector store from {index_path}...")
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    
    try:
        vectorstore = FAISS.load_local(
            index_path, 
            embeddings,
            allow_dangerous_deserialization=True
        )
        print(f"✓ Vector store loaded")
        return vectorstore
    except Exception as e:
        print(f"✗ Error loading vector store: {e}")
        return None

# --------------------------------------------------
# 6. Junk Chunk Filter
# --------------------------------------------------
def is_junk_chunk(text: str) -> bool:
    """Filter out low-quality chunks"""
    if len(text) < 200:  # Too short
        return True
    if text.count("@") > 2:  # Likely email/metadata
        return True
    if text.count("http") > 3:  # Mostly URLs
        return True
    return False

# --------------------------------------------------
# 7. Retrieve Context
# --------------------------------------------------
def retrieve_context(vectorstore, question, k=8, max_distance=1.0):
    """Retrieve and filter relevant chunks"""
    print(f"\nRetrieving context for: '{question}'")
    
    # Get candidates
    results = vectorstore.similarity_search_with_score(question, k=k)
    
    # Filter by quality and relevance
    filtered = []
    for doc, distance in results:
        # Lower distance = more similar (FAISS uses L2 distance)
        if distance > max_distance:  # Too far
            continue
        if is_junk_chunk(doc.page_content):
            continue
        filtered.append((doc, distance))
    
    # Sort by distance (best first) and take top chunks
    filtered.sort(key=lambda x: x[1])
    selected = filtered[:4]
    
    if not selected:
        print("⚠ Warning: No high-quality context retrieved")
        return []
    
    print(f"✓ Selected {len(selected)} relevant chunks")
    return selected

# --------------------------------------------------
# 8. Build Context String
# --------------------------------------------------
def build_context(selected_chunks):
    """Format chunks into numbered context blocks"""
    context_blocks = []
    for i, (doc, distance) in enumerate(selected_chunks, 1):
        context_blocks.append(
            f"[DOC {i}]\n{doc.page_content.strip()}"
        )
    return "\n\n".join(context_blocks)

# --------------------------------------------------
# 9. Generate Answer with Citations
# --------------------------------------------------
def generate_answer(question, context):
    """Generate answer with citation requirements"""
    print("Generating answer...")
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    messages = [
        (
            "system",
            "You answer questions using the provided documents. "
            "If a statement is supported by a document, cite it using [DOC X]. "
            "If the documents do not clearly support a statement, "
            "still answer but mark it as [UNSUPPORTED]."
        ),
        (
            "user",
            f"""Documents:
{context}

Question:
{question}

Rules:
- Answer in plain language
- Every factual statement must end with either:
  • a citation like [DOC 1]
  • or the tag [UNSUPPORTED]
- Do not refuse to answer
- Do not explain your reasoning
- Do not summarize documents

Answer:"""
        )
    ]
    
    response = llm.invoke(messages)
    print(f"✓ Answer generated\n")
    return response.content

# --------------------------------------------------
# 10. Extract Claims
# --------------------------------------------------
def extract_claims(text):
    """Extract factual claims from answer"""
    print("Extracting claims...")
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = f"""Extract ONLY factual claims about the world or the document content from the text below.

Definition:
A valid claim is a declarative statement that asserts a fact about:
- entities, events, dates, locations, or properties described in the document

Do NOT extract:
- statements about missing information
- statements about lack of evidence
- statements explaining why the answer is unsupported
- meta statements like "the answer is unsupported" or "the documents do not provide information"

Instructions:
- Return each valid claim on a separate line
- Keep claims atomic
- If there are NO valid factual claims, return ONLY the word: NONE

Text:
\"\"\"{text}\"\"\"
"""
    
    response = llm.invoke([{"role": "user", "content": prompt}])
    output = response.content.strip()
    
    if output == "NONE":
        print("✓ No factual claims found")
        return []
    
    claims = [c.strip() for c in output.split("\n") if c.strip()]
    print(f"✓ Extracted {len(claims)} claims")
    return claims

# --------------------------------------------------
# 11. Fact-Check Claims
# --------------------------------------------------
def fact_check_claim(claim, context):
    """Check if claim is supported by context"""
    verifier_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = f"""You are a strict fact-checker.

Claim:
{claim}

Context:
{context}

Task:
Determine whether the claim is supported by the context.

Rules:
- Use ONLY the information explicitly stated or clearly implied in the context
- Do NOT use outside knowledge
- Do NOT assume missing facts
- If the context is related but does not directly support the claim, return 0.5
- If the context clearly supports the claim, return 1.0
- If the context does not support the claim, return 0.0

Return ONLY a single number: 1.0, 0.5, or 0.0
"""
    
    response = verifier_llm.invoke([{"role": "user", "content": prompt}])
    
    try:
        score = float(response.content.strip())
        return max(0.0, min(1.0, score))  # Clamp to [0, 1]
    except ValueError:
        return 0.0

# --------------------------------------------------
# 12. Verify All Claims
# --------------------------------------------------
def verify_claims(claims, vectorstore):
    """Verify each claim against the vector store"""
    print("\n" + "="*80)
    print("FACT-CHECKING CLAIMS")
    print("="*80 + "\n")
    
    all_scores = []
    
    for i, claim in enumerate(claims, 1):
        print(f"{i}. {claim}")
        
        # Retrieve relevant chunks for this claim
        search_results = vectorstore.similarity_search_with_score(claim, k=3)
        
        # Check claim against each chunk
        scores = []
        for doc, _ in search_results:
            score = fact_check_claim(claim, doc.page_content)
            scores.append(score)
        
        # Take the maximum score (best supporting evidence)
        max_score = max(scores) if scores else 0.0
        all_scores.append(max_score)
        
        # Display result
        if max_score >= 1.0:
            print(f"   ✓ SUPPORTED (score: {max_score})")
        elif max_score >= 0.5:
            print(f"   ⚠ PARTIAL (score: {max_score})")
        else:
            print(f"   ✗ HALLUCINATED (score: {max_score})")
        print()
    
    # Summary
    if all_scores:
        avg_score = sum(all_scores) / len(all_scores)
        supported = sum(1 for s in all_scores if s >= 1.0)
        partial = sum(1 for s in all_scores if 0.5 <= s < 1.0)
        hallucinated = sum(1 for s in all_scores if s < 0.5)
        
        print("="*80)
        print("SUMMARY")
        print("="*80)
        print(f"Total claims: {len(all_scores)}")
        print(f"Supported: {supported} ({supported/len(all_scores)*100:.1f}%)")
        print(f"Partial: {partial} ({partial/len(all_scores)*100:.1f}%)")
        print(f"Hallucinated: {hallucinated} ({hallucinated/len(all_scores)*100:.1f}%)")
        print(f"Average score: {avg_score:.2f}")
    
    return all_scores

# --------------------------------------------------
# 13. Main Pipeline
# --------------------------------------------------
def main():
    print("\n" + "="*80)
    print("RAG PIPELINE WITH HALLUCINATION DETECTION")
    print("="*80 + "\n")
    
    # Configuration
    PDF_PATH = "2511.18538v5.pdf"
    INDEX_PATH = "faiss_index"
    QUESTION = "What are code foundation models and how do they work?"  # ← CHANGE QUESTION HERE
    
    # Check if we should reload or create new
    REBUILD_INDEX = False  # Set to True to rebuild from scratch
    
    # Step 1: Load or Create Vector Store
    if REBUILD_INDEX or not os.path.exists(INDEX_PATH):
        print("Building new vector store...\n")
        pages = load_document(PDF_PATH)
        if pages is None:
            return
        
        pages = clean_pages(pages)
        chunks = chunk_text(pages)
        vectorstore = create_vectorstore(chunks, INDEX_PATH)
    else:
        vectorstore = load_vectorstore(INDEX_PATH)
    
    if vectorstore is None:
        print("✗ Failed to load/create vector store")
        return
    
    # Step 2: Retrieve Context
    selected_chunks = retrieve_context(vectorstore, QUESTION, k=8, max_distance=1.0)
    
    if not selected_chunks:
        print("✗ Could not retrieve relevant context")
        return
    
    context = build_context(selected_chunks)
    
    # Step 3: Generate Answer
    answer = generate_answer(QUESTION, context)
    
    print("="*80)
    print("QUESTION")
    print("="*80)
    print(QUESTION)
    print()
    
    print("="*80)
    print("ANSWER")
    print("="*80)
    print(answer)
    print()
    
    # Step 4: Extract and Verify Claims
    claims = extract_claims(answer)
    
    if claims:
        verify_claims(claims, vectorstore)
    else:
        print("No claims to verify (answer may be purely unsupported)")

# --------------------------------------------------
# Run
# --------------------------------------------------
if __name__ == "__main__":
    main()


RAG PIPELINE WITH HALLUCINATION DETECTION

Loading vector store from faiss_index...
✓ Vector store loaded

Retrieving context for: 'What are code foundation models and how do they work?'
⚠ Warning: No high-quality context retrieved
✗ Could not retrieve relevant context


In [20]:
"""
Complete RAG Pipeline with Hallucination Detection
Load → Clean → Chunk → Embed → Retrieve → Answer → Fact-Check
"""

import re
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

# Load environment variables
load_dotenv()

# --------------------------------------------------
# 1. Load PDF
# --------------------------------------------------
def load_document(doc_path):
    try:
        loader = PyPDFLoader(doc_path)
        pages = loader.load()
        print(f"✓ Loaded {len(pages)} pages from {doc_path}")
        return pages 
    except Exception as e:
        print(f"✗ Error loading document: {e}")
        return None

# --------------------------------------------------
# 2. Clean Pages
# --------------------------------------------------
def clean_pages(pages):
    print("Cleaning pages...")
    for p in pages:
        text = p.page_content
        
        # Remove PDF metadata
        text = re.sub(r'producer:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creator:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creationdate:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'ptex\.fullbanner:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'trapped:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'arxivid:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'doi:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'license:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'title:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        
        # Remove author lists
        text = re.sub(r'author:\s*.*?(?=\n\n|\Z)', '', text, flags=re.IGNORECASE | re.DOTALL)
        
        # Remove arXiv identifiers
        text = re.sub(r'arXiv:\d+\.\d+v\d+\s+\[.*?\]\s+\d+\s+\w+\s+\d{4}', '', text)
        
        # Remove page numbers
        text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
        
        # Remove metadata blocks and separators
        text = re.sub(r"Metadata:\s*\{[^}]*\}", '', text)
        text = re.sub(r'-{40,}', '', text)
        text = re.sub(r'={40,}', '', text)
        
        # Normalize whitespace
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r' \n', '\n', text)
        
        p.page_content = text.strip()
    
    # Filter short pages
    pages = [p for p in pages if len(p.page_content) > 100]
    print(f"✓ Cleaned {len(pages)} pages")
    return pages

# --------------------------------------------------
# 3. Clean Individual Chunks
# --------------------------------------------------
def clean_chunk_content(text):
    # Remove chunk artifacts
    text = re.sub(r'CHUNK\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'chunk_id[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'char_len[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(page|page_label|source|total_pages)[:\s]+[^\s,}]+', '', text)
    text = re.sub(r'[-=]{4,}', '', text)
    text = re.sub(r'\[\d+\]\s*$', '', text)
    
    # Final cleanup
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# --------------------------------------------------
# 4. Chunk Text
# --------------------------------------------------
def chunk_text(pages, chunk_size=800, chunk_overlap=100):
    print(f"Chunking text (size={chunk_size}, overlap={chunk_overlap})...")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = text_splitter.split_documents(pages)
    
    # Clean and add metadata
    for i, c in enumerate(chunks):
        c.page_content = clean_chunk_content(c.page_content)
        c.metadata["chunk_id"] = i
        c.metadata["char_len"] = len(c.page_content)
        c.metadata["word_count"] = len(c.page_content.split())
    
    # Filter low-quality chunks
    chunks = [c for c in chunks if 
              len(c.page_content) > 50 and 
              not c.page_content.lower().startswith('metadata') and
              len(c.page_content.split()) > 5]
    
    print(f"✓ Created {len(chunks)} chunks")
    return chunks

# --------------------------------------------------
# 5. Create or Load Vector Store
# --------------------------------------------------
def create_vectorstore(chunks, index_path="faiss_index"):
    """Create new vector store from chunks"""
    print("Creating embeddings and vector store...")
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    
    vectorstore.save_local(index_path)
    print(f"✓ Vector store saved to {index_path}")
    return vectorstore

def load_vectorstore(index_path="faiss_index"):
    """Load existing vector store"""
    print(f"Loading vector store from {index_path}...")
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    
    try:
        vectorstore = FAISS.load_local(
            index_path, 
            embeddings,
            allow_dangerous_deserialization=True
        )
        print(f"✓ Vector store loaded")
        return vectorstore
    except Exception as e:
        print(f"✗ Error loading vector store: {e}")
        return None

# --------------------------------------------------
# 6. Junk Chunk Filter
# --------------------------------------------------
def is_junk_chunk(text: str) -> bool:
    """Filter out low-quality chunks"""
    if len(text) < 200:  # Too short
        return True
    if text.count("@") > 2:  # Likely email/metadata
        return True
    if text.count("http") > 3:  # Mostly URLs
        return True
    return False

# --------------------------------------------------
# 7. Retrieve Context
# --------------------------------------------------
def retrieve_context(vectorstore, question, k=8, max_distance=10.0):
    """Retrieve and filter relevant chunks"""
    print(f"\nRetrieving context for: '{question}'")
    
    # Get candidates
    results = vectorstore.similarity_search_with_score(question, k=k)
    
    print(f"  Retrieved {len(results)} candidates")
    if results:
        print(f"  Distance range: {results[0][1]:.3f} to {results[-1][1]:.3f}")
    
    # Filter by quality and relevance
    filtered = []
    for doc, distance in results:
        # Lower distance = more similar (FAISS uses L2 distance)
        if distance > max_distance:  # Too far
            continue
        if is_junk_chunk(doc.page_content):
            continue
        filtered.append((doc, distance))
    
    # Sort by distance (best first) and take top chunks
    filtered.sort(key=lambda x: x[1])
    selected = filtered[:4]
    
    if not selected:
        print("⚠ Warning: No high-quality context retrieved")
        return []
    
    print(f"✓ Selected {len(selected)} relevant chunks")
    return selected

# --------------------------------------------------
# 8. Build Context String
# --------------------------------------------------
def build_context(selected_chunks):
    """Format chunks into numbered context blocks"""
    context_blocks = []
    for i, (doc, distance) in enumerate(selected_chunks, 1):
        context_blocks.append(
            f"[DOC {i}]\n{doc.page_content.strip()}"
        )
    return "\n\n".join(context_blocks)

# --------------------------------------------------
# 9. Generate Answer with Citations
# --------------------------------------------------
def generate_answer(question, context):
    """Generate answer with citation requirements"""
    print("Generating answer...")
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    messages = [
        (
            "system",
            "You answer questions using the provided documents. "
            "If a statement is supported by a document, cite it using [DOC X]. "
            "If the documents do not clearly support a statement, "
            "still answer but mark it as [UNSUPPORTED]."
        ),
        (
            "user",
            f"""Documents:
{context}

Question:
{question}

Rules:
- Answer in plain language
- Every factual statement must end with either:
  • a citation like [DOC 1]
  • or the tag [UNSUPPORTED]
- Do not refuse to answer
- Do not explain your reasoning
- Do not summarize documents

Answer:"""
        )
    ]
    
    response = llm.invoke(messages)
    print(f"✓ Answer generated\n")
    return response.content

# --------------------------------------------------
# 10. Extract Claims
# --------------------------------------------------
def extract_claims(text):
    """Extract factual claims from answer"""
    print("Extracting claims...")
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = f"""Extract ONLY factual claims about the world or the document content from the text below.

Definition:
A valid claim is a declarative statement that asserts a fact about:
- entities, events, dates, locations, or properties described in the document

Do NOT extract:
- statements about missing information
- statements about lack of evidence
- statements explaining why the answer is unsupported
- meta statements like "the answer is unsupported" or "the documents do not provide information"

Instructions:
- Return each valid claim on a separate line
- Keep claims atomic
- If there are NO valid factual claims, return ONLY the word: NONE

Text:
\"\"\"{text}\"\"\"
"""
    
    response = llm.invoke([{"role": "user", "content": prompt}])
    output = response.content.strip()
    
    if output == "NONE":
        print("✓ No factual claims found")
        return []
    
    claims = [c.strip() for c in output.split("\n") if c.strip()]
    print(f"✓ Extracted {len(claims)} claims")
    return claims

# --------------------------------------------------
# 11. Fact-Check Claims
# --------------------------------------------------
def fact_check_claim(claim, context):
    """Check if claim is supported by context"""
    verifier_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = f"""You are a strict fact-checker.

Claim:
{claim}

Context:
{context}

Task:
Determine whether the claim is supported by the context.

Rules:
- Use ONLY the information explicitly stated or clearly implied in the context
- Do NOT use outside knowledge
- Do NOT assume missing facts
- If the context is related but does not directly support the claim, return 0.5
- If the context clearly supports the claim, return 1.0
- If the context does not support the claim, return 0.0

Return ONLY a single number: 1.0, 0.5, or 0.0
"""
    
    response = verifier_llm.invoke([{"role": "user", "content": prompt}])
    
    try:
        score = float(response.content.strip())
        return max(0.0, min(1.0, score))  # Clamp to [0, 1]
    except ValueError:
        return 0.0

# --------------------------------------------------
# 12. Verify All Claims
# --------------------------------------------------
def verify_claims(claims, vectorstore):
    """Verify each claim against the vector store"""
    print("\n" + "="*80)
    print("FACT-CHECKING CLAIMS")
    print("="*80 + "\n")
    
    all_scores = []
    
    for i, claim in enumerate(claims, 1):
        print(f"{i}. {claim}")
        
        # Retrieve relevant chunks for this claim
        search_results = vectorstore.similarity_search_with_score(claim, k=3)
        
        # Check claim against each chunk
        scores = []
        for doc, _ in search_results:
            score = fact_check_claim(claim, doc.page_content)
            scores.append(score)
        
        # Take the maximum score (best supporting evidence)
        max_score = max(scores) if scores else 0.0
        all_scores.append(max_score)
        
        # Display result
        if max_score >= 1.0:
            print(f"   ✓ SUPPORTED (score: {max_score})")
        elif max_score >= 0.5:
            print(f"   ⚠ PARTIAL (score: {max_score})")
        else:
            print(f"   ✗ HALLUCINATED (score: {max_score})")
        print()
    
    # Summary
    if all_scores:
        avg_score = sum(all_scores) / len(all_scores)
        supported = sum(1 for s in all_scores if s >= 1.0)
        partial = sum(1 for s in all_scores if 0.5 <= s < 1.0)
        hallucinated = sum(1 for s in all_scores if s < 0.5)
        
        print("="*80)
        print("SUMMARY")
        print("="*80)
        print(f"Total claims: {len(all_scores)}")
        print(f"Supported: {supported} ({supported/len(all_scores)*100:.1f}%)")
        print(f"Partial: {partial} ({partial/len(all_scores)*100:.1f}%)")
        print(f"Hallucinated: {hallucinated} ({hallucinated/len(all_scores)*100:.1f}%)")
        print(f"Average score: {avg_score:.2f}")
    
    return all_scores

# --------------------------------------------------
# 13. Main Pipeline
# --------------------------------------------------
def main():
    print("\n" + "="*80)
    print("RAG PIPELINE WITH HALLUCINATION DETECTION")
    print("="*80 + "\n")
    
    # Configuration
    PDF_PATH = "2511.18538v5.pdf"
    INDEX_PATH = "code_intelligence_index"
    QUESTION =  "What is the main focus and contribution of this survey paper?" # ← CHANGE QUESTION HERE
    
    # Check if we should reload or create new
    REBUILD_INDEX = False  # Set to True to rebuild from scratch
    
    # Step 1: Load or Create Vector Store
    if REBUILD_INDEX or not os.path.exists(INDEX_PATH):
        print("Building new vector store...\n")
        pages = load_document(PDF_PATH)
        if pages is None:
            return
        
        pages = clean_pages(pages)
        chunks = chunk_text(pages)
        vectorstore = create_vectorstore(chunks, INDEX_PATH)
    else:
        vectorstore = load_vectorstore(INDEX_PATH)
    
    if vectorstore is None:
        print("✗ Failed to load/create vector store")
        return
    
    # Step 2: Retrieve Context
    selected_chunks = retrieve_context(vectorstore, QUESTION, k=8, max_distance=10.0)
    
    if not selected_chunks:
        print("✗ Could not retrieve relevant context")
        return
    
    context = build_context(selected_chunks)
    
    # Step 3: Generate Answer
    answer = generate_answer(QUESTION, context)
    
    print("="*80)
    print("QUESTION")
    print("="*80)
    print(QUESTION)
    print()
    
    print("="*80)
    print("ANSWER")
    print("="*80)
    print(answer)
    print()
    
    # Step 4: Extract and Verify Claims
    claims = extract_claims(answer)
    
    if claims:
        verify_claims(claims, vectorstore)
    else:
        print("No claims to verify (answer may be purely unsupported)")

# --------------------------------------------------
# Run
# --------------------------------------------------
if __name__ == "__main__":
    main()


RAG PIPELINE WITH HALLUCINATION DETECTION

Loading vector store from code_intelligence_index...
✓ Vector store loaded

Retrieving context for: 'What is the main focus and contribution of this survey paper?'
  Retrieved 8 candidates
  Distance range: 1.077 to 1.262
✓ Selected 4 relevant chunks
Generating answer...
✓ Answer generated

QUESTION
What is the main focus and contribution of this survey paper?

ANSWER
The main focus of this survey paper is to provide a comprehensive and contemporary synthesis of research literature on large language models (LLMs) for code intelligence. It aims to systematically examine the entire model life cycle, including aspects like code pre-training, supervised fine-tuning, and reinforcement learning. The paper also discusses evaluation methodologies, emerging trends, and open challenges in the field, serving as a reference for researchers and a roadmap for practitioners seeking to use these technologies in production environments [DOC 4].

Extracting cl

In [1]:
"""
Interactive RAG Pipeline with GUI File Upload
Users can upload their own PDFs through a file browser
"""

import re
import os
import tkinter as tk
from tkinter import filedialog
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

# Load environment variables
load_dotenv()

# --------------------------------------------------
# Document Processing Functions
# --------------------------------------------------
def load_document(doc_path):
    try:
        loader = PyPDFLoader(doc_path)
        pages = loader.load()
        print(f"✓ Loaded {len(pages)} pages from {os.path.basename(doc_path)}")
        return pages 
    except Exception as e:
        print(f"✗ Error loading document: {e}")
        return None

def clean_pages(pages):
    print("Cleaning pages...")
    for p in pages:
        text = p.page_content
        
        # Remove PDF metadata
        text = re.sub(r'producer:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creator:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'creationdate:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'ptex\.fullbanner:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'trapped:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'arxivid:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'doi:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'license:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'title:.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
        
        # Remove author lists
        text = re.sub(r'author:\s*.*?(?=\n\n|\Z)', '', text, flags=re.IGNORECASE | re.DOTALL)
        
        # Remove arXiv identifiers
        text = re.sub(r'arXiv:\d+\.\d+v\d+\s+\[.*?\]\s+\d+\s+\w+\s+\d{4}', '', text)
        
        # Remove page numbers
        text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
        
        # Remove metadata blocks and separators
        text = re.sub(r"Metadata:\s*\{[^}]*\}", '', text)
        text = re.sub(r'-{40,}', '', text)
        text = re.sub(r'={40,}', '', text)
        
        # Normalize whitespace
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r' \n', '\n', text)
        
        p.page_content = text.strip()
    
    pages = [p for p in pages if len(p.page_content) > 100]
    print(f"✓ Cleaned {len(pages)} pages")
    return pages

def clean_chunk_content(text):
    text = re.sub(r'CHUNK\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'chunk_id[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'char_len[:\s]+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(page|page_label|source|total_pages)[:\s]+[^\s,}]+', '', text)
    text = re.sub(r'[-=]{4,}', '', text)
    text = re.sub(r'\[\d+\]\s*$', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def chunk_text(pages, chunk_size=800, chunk_overlap=100):
    print(f"Chunking text (size={chunk_size}, overlap={chunk_overlap})...")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = text_splitter.split_documents(pages)
    
    for i, c in enumerate(chunks):
        c.page_content = clean_chunk_content(c.page_content)
        c.metadata["chunk_id"] = i
        c.metadata["char_len"] = len(c.page_content)
        c.metadata["word_count"] = len(c.page_content.split())
    
    chunks = [c for c in chunks if 
              len(c.page_content) > 50 and 
              not c.page_content.lower().startswith('metadata') and
              len(c.page_content.split()) > 5]
    
    print(f"✓ Created {len(chunks)} chunks")
    return chunks

def create_vectorstore(chunks, index_path="faiss_index"):
    print("Creating embeddings and vector store...")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
    vectorstore.save_local(index_path)
    print(f"✓ Vector store saved to {index_path}")
    return vectorstore

def load_vectorstore(index_path="faiss_index"):
    print(f"Loading vector store from {index_path}...")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    try:
        vectorstore = FAISS.load_local(
            index_path, 
            embeddings,
            allow_dangerous_deserialization=True
        )
        print(f"✓ Vector store loaded")
        return vectorstore
    except Exception as e:
        print(f"✗ Error loading vector store: {e}")
        return None

def is_junk_chunk(text: str) -> bool:
    if len(text) < 100:
        return True
    if text.count("@") > 5:
        return True
    if text.count("http") > 5:
        return True
    return False

def retrieve_context(vectorstore, question, k=8, max_distance=2.0):
    results = vectorstore.similarity_search_with_score(question, k=k)
    
    # Debug: show what we got
    print(f"   Retrieved {len(results)} chunks from vector store")
    if results:
        print(f"   Distance range: {results[0][1]:.3f} to {results[-1][1]:.3f}")
    
    filtered = []
    for doc, distance in results:
        if distance > max_distance:
            continue
        if is_junk_chunk(doc.page_content):
            continue
        filtered.append((doc, distance))
    
    filtered.sort(key=lambda x: x[1])
    selected = filtered[:4]
    
    print(f"   Selected {len(selected)} relevant chunks after filtering")
    
    return selected

def build_context(selected_chunks):
    context_blocks = []
    for i, (doc, distance) in enumerate(selected_chunks, 1):
        context_blocks.append(f"[DOC {i}]\n{doc.page_content.strip()}")
    return "\n\n".join(context_blocks)

def generate_answer(question, context):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    messages = [
        (
            "system",
            "You answer questions using the provided documents. "
            "If a statement is supported by a document, cite it using [DOC X]. "
            "If the documents do not clearly support a statement, "
            "still answer but mark it as [UNSUPPORTED]."
        ),
        (
            "user",
            f"""Documents:
{context}

Question:
{question}

Rules:
- Answer in plain language
- Every factual statement must end with either:
  • a citation like [DOC 1]
  • or the tag [UNSUPPORTED]
- Do not refuse to answer
- Do not explain your reasoning
- Do not summarize documents

Answer:"""
        )
    ]
    
    response = llm.invoke(messages)
    return response.content

def extract_claims(text):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = f"""Extract ALL factual claims from the text below.

A factual claim is any statement that:
- Describes what something is or does
- States a property, feature, or capability
- Mentions a number, measurement, or comparison
- Describes how something works
- States a relationship between entities

Extract each claim as a separate sentence.

Text:
\"\"\"{text}\"\"\"

Instructions:
- Extract EVERY factual statement
- Keep each claim as a complete sentence
- One claim per line
- If there are no factual claims, return: NONE

Claims:"""
    
    response = llm.invoke([{"role": "user", "content": prompt}])
    output = response.content.strip()
    
    if output == "NONE" or not output:
        return []
    
    # Split by newlines and clean
    claims = []
    for line in output.split("\n"):
        line = line.strip()
        # Remove numbering like "1. ", "- ", etc.
        line = re.sub(r'^\d+\.\s*', '', line)
        line = re.sub(r'^[-•*]\s*', '', line)
        if line and len(line) > 10:  # Ignore very short lines
            claims.append(line)
    
    return claims

def fact_check_claim(claim, context):
    verifier_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = f"""You are a strict fact-checker.

Claim:
{claim}

Context:
{context}

Task:
Determine whether the claim is supported by the context.

Rules:
- Use ONLY the information explicitly stated or clearly implied in the context
- Do NOT use outside knowledge
- Do NOT assume missing facts
- If the context is related but does not directly support the claim, return 0.5
- If the context clearly supports the claim, return 1.0
- If the context does not support the claim, return 0.0

Return ONLY a single number: 1.0, 0.5, or 0.0
"""
    
    response = verifier_llm.invoke([{"role": "user", "content": prompt}])
    
    try:
        score = float(response.content.strip())
        return max(0.0, min(1.0, score))
    except ValueError:
        return 0.0

def verify_claims(claims, vectorstore, verbose=False):
    """Verify claims and return results"""
    results = []
    
    for claim in claims:
        search_results = vectorstore.similarity_search_with_score(claim, k=3)
        
        scores = []
        for doc, _ in search_results:
            score = fact_check_claim(claim, doc.page_content)
            scores.append(score)
        
        max_score = max(scores) if scores else 0.0
        
        # Categorize
        if max_score >= 1.0:
            status = "SUPPORTED"
        elif max_score >= 0.5:
            status = "PARTIAL"
        else:
            status = "HALLUCINATED"
        
        results.append({
            "claim": claim,
            "score": max_score,
            "status": status
        })
        
        if verbose:
            print(f"• {claim}")
            print(f"  → {status} (score: {max_score})\n")
    
    return results

# --------------------------------------------------
# Hallucination Report
# --------------------------------------------------
def generate_hallucination_report(results):
    """Generate a clean hallucination report"""
    if not results:
        return {
            "total_claims": 0,
            "supported": 0,
            "partial": 0,
            "hallucinated": 0,
            "accuracy_score": 0.0,
            "reliability": "NO CLAIMS"
        }
    
    total = len(results)
    supported = sum(1 for r in results if r["status"] == "SUPPORTED")
    partial = sum(1 for r in results if r["status"] == "PARTIAL")
    hallucinated = sum(1 for r in results if r["status"] == "HALLUCINATED")
    
    avg_score = sum(r["score"] for r in results) / total
    
    # Determine reliability level
    if avg_score >= 0.9:
        reliability = "EXCELLENT"
    elif avg_score >= 0.7:
        reliability = "GOOD"
    elif avg_score >= 0.5:
        reliability = "FAIR"
    else:
        reliability = "POOR"
    
    return {
        "total_claims": total,
        "supported": supported,
        "partial": partial,
        "hallucinated": hallucinated,
        "accuracy_score": avg_score,
        "reliability": reliability,
        "details": results
    }

def print_hallucination_report(report):
    """Print a formatted hallucination report"""
    print("\n" + "="*80)
    print("📊 HALLUCINATION REPORT")
    print("="*80)
    
    if report["total_claims"] == 0:
        print("No factual claims detected in the answer.")
        print("="*80 + "\n")
        return
    
    total = report["total_claims"]
    supported = report["supported"]
    partial = report["partial"]
    hallucinated = report["hallucinated"]
    score = report["accuracy_score"]
    reliability = report["reliability"]
    
    # Summary stats
    print(f"\n📈 Summary:")
    print(f"   Total Claims: {total}")
    print(f"   ✓ Supported: {supported} ({supported/total*100:.1f}%)")
    print(f"   ⚠ Partial: {partial} ({partial/total*100:.1f}%)")
    print(f"   ✗ Hallucinated: {hallucinated} ({hallucinated/total*100:.1f}%)")
    print(f"\n   Accuracy Score: {score:.2f}/1.00")
    print(f"   Reliability: {reliability}")
    
    # Detailed breakdown
    print(f"\n📋 Claim-by-Claim Analysis:")
    print("-" * 80)
    
    for i, detail in enumerate(report["details"], 1):
        status_symbol = {
            "SUPPORTED": "✓",
            "PARTIAL": "⚠",
            "HALLUCINATED": "✗"
        }[detail["status"]]
        
        print(f"\n{i}. {detail['claim']}")
        print(f"   {status_symbol} {detail['status']} (confidence: {detail['score']:.2f})")
    
    print("\n" + "="*80 + "\n")

# --------------------------------------------------
# File Upload Dialog
# --------------------------------------------------
def upload_pdf_file():
    """Open file dialog to select PDF"""
    print("\n" + "="*80)
    print("📤 UPLOAD PDF FILE")
    print("="*80)
    print("\nOpening file browser... Please select a PDF file.")
    
    # Create a hidden root window
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    root.attributes('-topmost', True)  # Bring dialog to front
    
    # Open file dialog
    file_path = filedialog.askopenfilename(
        title="Select a PDF file",
        filetypes=[
            ("PDF files", "*.pdf"),
            ("All files", "*.*")
        ]
    )
    
    root.destroy()  # Close the hidden window
    
    if file_path:
        print(f"\n✓ Selected: {os.path.basename(file_path)}")
        print(f"   Path: {file_path}")
        return file_path
    else:
        print("\n✗ No file selected")
        return None

def get_index_path(pdf_path):
    """Generate unique index path for each PDF"""
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    clean_name = re.sub(r'[^\w\-]', '_', pdf_name)
    return f"faiss_index_{clean_name}"

# --------------------------------------------------
# Interactive Q&A Loop
# --------------------------------------------------
def interactive_qa(vectorstore, pdf_name):
    """Interactive question-answering with hallucination detection"""
    print("\n" + "="*80)
    print("🤖 INTERACTIVE RAG SYSTEM")
    print("="*80)
    print(f"📄 Current document: {pdf_name}")
    print("\nAsk questions about the document. Type 'quit' or 'exit' to stop.")
    print("="*80 + "\n")
    
    while True:
        # Get user question
        question = input("\n❓ Your Question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Goodbye!\n")
            break
        
        if not question:
            print("⚠ Please enter a question.\n")
            continue
        
        print("\n⏳ Processing...")
        
        # Retrieve context
        selected_chunks = retrieve_context(vectorstore, question, k=8, max_distance=2.0)
        
        if not selected_chunks:
            print(f"⚠ Could not retrieve relevant context for: '{question}'")
            print("   Try rephrasing your question or asking about a different topic.\n")
            continue
        
        context = build_context(selected_chunks)
        
        # Generate answer
        answer = generate_answer(question, context)
        
        # Display answer
        print("\n" + "="*80)
        print("💬 ANSWER")
        print("="*80)
        print(answer)
        
        # Extract and verify claims
        print("\n⏳ Extracting claims...")
        claims = extract_claims(answer)
        print(f"   Found {len(claims)} claims")
        
        if claims:
            print("⏳ Verifying claims...")
            results = verify_claims(claims, vectorstore, verbose=False)
            report = generate_hallucination_report(results)
            print_hallucination_report(report)
        else:
            print("\n" + "="*80)
            print("📊 HALLUCINATION REPORT")
            print("="*80)
            print("No factual claims detected in the answer.")
            print("="*80 + "\n")

# --------------------------------------------------
# Vector Store Setup
# --------------------------------------------------
def setup_vectorstore(pdf_path, index_path, rebuild=False):
    """Setup vector store (load or create)"""
    if rebuild or not os.path.exists(index_path):
        print("\n🔨 Building new vector store...\n")
        pages = load_document(pdf_path)
        if pages is None:
            return None
        
        pages = clean_pages(pages)
        chunks = chunk_text(pages)
        vectorstore = create_vectorstore(chunks, index_path)
    else:
        vectorstore = load_vectorstore(index_path)
    
    return vectorstore

# --------------------------------------------------
# Main Entry Point
# --------------------------------------------------
def main():
    print("\n" + "="*80)
    print("🚀 RAG SYSTEM WITH HALLUCINATION DETECTION")
    print("="*80 + "\n")
    
    # Upload PDF file
    pdf_path = upload_pdf_file()
    
    if pdf_path is None:
        print("\n✗ No PDF selected. Exiting.")
        return
    
    # Generate index path based on PDF name
    index_path = get_index_path(pdf_path)
    
    # Check if index already exists
    if os.path.exists(index_path):
        print(f"\n📦 Found existing index for this PDF: {index_path}")
        rebuild = input("   Rebuild index? (y/N): ").strip().lower()
        rebuild_index = rebuild == 'y'
    else:
        print(f"\n🔨 No existing index found. Will create new index: {index_path}")
        rebuild_index = True
    
    # Setup vector store
    vectorstore = setup_vectorstore(pdf_path, index_path, rebuild_index)
    
    if vectorstore is None:
        print("✗ Failed to load/create vector store")
        return
    
    print("\n✓ System ready!\n")
    
    # Start interactive loop
    pdf_name = os.path.basename(pdf_path)
    interactive_qa(vectorstore, pdf_name)

if __name__ == "__main__":
    main()


🚀 RAG SYSTEM WITH HALLUCINATION DETECTION


📤 UPLOAD PDF FILE

Opening file browser... Please select a PDF file.


: 